In [ ]:
import os
import json
import time
import requests
from requests.auth import HTTPBasicAuth

from eodag import EODataAccessGateway


Define variables:

In [37]:
GRASS_PROJECT = "s2_proj_epsg32635"
START_YEAR = 2025
START_MONTH = 8
END_YEAR = 2025
END_MONTH = 8
TILE_ID = "35SKC"
MAIN_PROCESS_CHAIN = "pc_MAIN.json"


Define actinia variables:

In [ ]:
# variables to set the actinia host, version, and user
actinia_baseurl = "http://localhost:8088"
actinia_version = "v3"
actinia_url = actinia_baseurl + "/api/" + actinia_version
actinia_auth = HTTPBasicAuth('actinia', 'actinia')  # username, password


### Filter Sentinel-2 scenes

In [4]:
search_criteria = {
    "productType": "S2MSI2A",
    "start": f"{START_YEAR}-{START_MONTH:02d}-01",
    "end": f"{END_YEAR}-{END_MONTH:02d}-05",
    "tileIdentifier": TILE_ID,
}


In [35]:
dag = EODataAccessGateway()
all_products = dag.search_all(**search_criteria)
s2_ids = [all_products[i].properties['id']  for i in range(len(all_products))]
print(f"Found <{len(s2_ids)}> Sentinel-2 scene IDs:")
for id in s2_ids:
    print(f" - {id}")


Found <2> Sentinel-2 scene IDs:
 - S2C_MSIL2A_20250802T090611_N0511_R050_T35SKC_20250802T124515
 - S2A_MSIL2A_20250804T091331_N0511_R050_T35SKC_20250804T120007


### Start actinia

In [6]:
# helper function to print formatted JSON using the json module

def print_as_json(data):
    print(json.dumps(data, indent=2))

# helper function to verify a request
def verify_request(request, success_code=200):
    if request.status_code != success_code:
        print("ERROR: actinia processing failed with status code %d!" % request.status_code)
        print("See errors below:")
        print_as_json(request.json())
        request_url = request.json()["urls"]["status"]
        requests.delete(url=request_url, auth=actinia_auth)
        raise Exception("The resource <%s> has been terminated." % request_url)

Construct actinia endpoint:
`...locations/{GRASS_PROJECT}/mapsets/PERMANENT/processing` -> persistent, data is kept, needs
`...locations/{GRASS_PROJECT}/processing_export` -> non-persistent, data is not kept in GRASSDB

In [ ]:
# make a POST request to the actinia data API
request_url = f"{actinia_url}/locations/{GRASS_PROJECT}/processing_export"
# request_url = f"{actinia_url}/locations/{GRASS_PROJECT}/mapsets/PERMANENT/processing_export"

print("actinia POST request:")
print(request_url)
print("---")

actinia POST request:
http://localhost:8088/api/v3/locations/s2_proj_epsg32635/processing_export
---


In [ ]:
# loop over all Sentinel-2 scene IDs and start processing
# list for storing status request URLs
request_url_lst = []
for s2_id in s2_ids:
    print(f"Processing Sentinel-2 scene ID: {s2_id}")
    # load the main process chain
    process_chain = json.load(open(MAIN_PROCESS_CHAIN))
    # modify the process chain with the current Sentinel-2 scene ID
    process_chain["list"][0]["inputs"][0]["value"] = s2_id
    # modify the output names for NDVI and NDWI
    process_chain["list"][4]["inputs"][2]['value'] = f"NDVI_{s2_id.split('_')[2][:8]}_{s2_id.split('_')[0]}"
    process_chain["list"][5]["inputs"][2]['value'] = f"NDWI_{s2_id.split('_')[2][:8]}_{s2_id.split('_')[0]}"

    # make the POST request to start the processing
    request = requests.post(url=request_url, auth=actinia_auth, json=process_chain)
    # check if anything went wrong
    verify_request(request, 200)
    # get a json-encoded content of the response
    jsonResponse = request.json()
    # save the status request URL
    request_url_lst.append(jsonResponse["urls"]["status"])

    while jsonResponse["status"] in ("accepted", "running"):
        status_request_url = jsonResponse["urls"]["status"]
        status_request = requests.get(url=status_request_url.replace("https", "http"), auth=actinia_auth)
        jsonResponse = status_request.json()
        print(f"Polling status for {s2_id}: {jsonResponse['status']}")
        print(f"Doing: {jsonResponse['message']}")
        print('#########################################')
        time.sleep(30)

    print(f"Final status for {s2_id}: {jsonResponse['status']}")
    print(f"Processing for {s2_id} finished.")
    print('=========================================')




Processing Sentinel-2 scene ID: S2C_MSIL2A_20250802T090611_N0511_R050_T35SKC_20250802T124515
Polling status for S2C_MSIL2A_20250802T090611_N0511_R050_T35SKC_20250802T124515: accepted
Doing: Resource accepted
#########################################
Polling status for S2C_MSIL2A_20250802T090611_N0511_R050_T35SKC_20250802T124515: running
Doing: Running executable i.s2_id.download with parameters ['s2_id=S2C_MSIL2A_20250802T090611_N0511_R050_T35SKC_20250802T124515', 'download_dir=/home/data/'] for 30.137610912322998 seconds
#########################################
Polling status for S2C_MSIL2A_20250802T090611_N0511_R050_T35SKC_20250802T124515: running
Doing: Running executable i.s2_id.download with parameters ['s2_id=S2C_MSIL2A_20250802T090611_N0511_R050_T35SKC_20250802T124515', 'download_dir=/home/data/'] for 60.26830768585205 seconds
#########################################
Polling status for S2C_MSIL2A_20250802T090611_N0511_R050_T35SKC_20250802T124515: running
Doing: Running executa

Print links to tif of NDVI/NDWI results

In [32]:
# print links to processed tifs
for s2_id in request_url_lst:
    status_request = requests.get(url=s2_id.replace("https", "http"), auth=actinia_auth)
    jsonResponse = status_request.json()
    for tif in jsonResponse['urls']['resources']:
        print(f"{os.path.basename(tif)}: {tif.replace('https', 'http')}")

NDVI_20250802_S2C.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-083c57a6-0379-4968-b195-7aaf303e7f89/NDVI_20250802_S2C.tif
NDWI_20250802_S2C.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-083c57a6-0379-4968-b195-7aaf303e7f89/NDWI_20250802_S2C.tif
NDVI_20250804_S2A.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-bb8eaae4-b8c3-48ba-bb73-4832146ecccc/NDVI_20250804_S2A.tif
NDWI_20250804_S2A.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-bb8eaae4-b8c3-48ba-bb73-4832146ecccc/NDWI_20250804_S2A.tif
